In [39]:
import pandas as pd
import glob
import sqlite3
from scipy import stats


# ============================================================
# 1. Load and combine five EPL seasons
# ============================================================

cols = [
    'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'HS', 'AS', 'HST', 'AST',
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

files = sorted(glob.glob('Seasons/E0_*.csv'))

dfs = []

for f in files:
    season = f.split('E0_')[1].replace('.csv', '')
    
    df = pd.read_csv(f)[cols]
    df['Season'] = season
    
    dfs.append(df)

all_seasons = pd.concat(dfs, ignore_index=True)

print(f"Loaded {len(all_seasons)} matches")
print(f"Seasons: {all_seasons['Season'].nunique()}")


# ============================================================
# 2. Calculate Home vs Away Points Per Game
# ============================================================

all_seasons['HomePoints'] = all_seasons['FTR'].map({
    'H': 3,
    'D': 1,
    'A': 0
})

all_seasons['AwayPoints'] = all_seasons['FTR'].map({
    'A': 3,
    'D': 1,
    'H': 0
})


home = (
    all_seasons
    .groupby(['Season', 'HomeTeam'])['HomePoints']
    .mean()
    .reset_index()
    .rename(columns={
        'HomeTeam': 'Team',
        'HomePoints': 'HomePPG'
    })
)

away = (
    all_seasons
    .groupby(['Season', 'AwayTeam'])['AwayPoints']
    .mean()
    .reset_index()
    .rename(columns={
        'AwayTeam': 'Team',
        'AwayPoints': 'AwayPPG'
    })
)


ppg = home.merge(
    away,
    on=['Season', 'Team']
)

ppg['PPG_Diff'] = ppg['HomePPG'] - ppg['AwayPPG']


# ============================================================
# 3. Statistical Test
# ============================================================

print("\nAverage PPG:")
print(ppg[['HomePPG', 'AwayPPG']].mean())

t_stat, p_value = stats.ttest_rel(
    ppg['HomePPG'],
    ppg['AwayPPG']
)

print(f"\nt-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.5f}")

extra_pts_per_season = (
    ppg['HomePPG'].mean() -
    ppg['AwayPPG'].mean()
) * 19

print(
    f"Estimated home advantage: "
    f"~{extra_pts_per_season:.1f} points per season"
)


# ============================================================
# 4. Store data in SQLite
# ============================================================

conn = sqlite3.connect('epl.db')

all_seasons.to_sql(
    'matches',
    conn,
    if_exists='replace',
    index=False
)


# ============================================================
# 5. SQL Analysis
# ============================================================

query = """
WITH home_ppg AS (
    SELECT
        Season,
        HomeTeam AS Team,
        AVG(
            CASE
                WHEN FTR = 'H' THEN 3
                WHEN FTR = 'D' THEN 1
                ELSE 0
            END
        ) AS HomePPG
    FROM matches
    GROUP BY Season, HomeTeam
),

away_ppg AS (
    SELECT
        Season,
        AwayTeam AS Team,
        AVG(
            CASE
                WHEN FTR = 'A' THEN 3
                WHEN FTR = 'D' THEN 1
                ELSE 0
            END
        ) AS AwayPPG
    FROM matches
    GROUP BY Season, AwayTeam
)

SELECT
    h.Season,
    h.Team,
    h.HomePPG,
    a.AwayPPG,
    h.HomePPG - a.AwayPPG AS PPG_Diff

FROM home_ppg h

JOIN away_ppg a
    ON h.Season = a.Season
    AND h.Team = a.Team

ORDER BY h.Season, h.Team;
"""

result = pd.read_sql(query, conn)

result.to_csv(
    'Seasons/ppg_by_team_season.csv',
    index=False
)

conn.close()

Loaded 1900 matches
Seasons: 5

Average PPG:
HomePPG    1.563684
AwayPPG    1.197368
dtype: float64

t-statistic: 9.741
p-value: 0.00000
Estimated home advantage: ~7.0 points per season
